In [1]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
load_dotenv()
import os
from tqdm.auto import tqdm

db_host = os.environ["DB_HOST"]
db_port = os.environ["DB_PORT"]
db_name = os.environ["DB_NAME"]
db_pw = os.environ["DB_PASSWORD"]
db_us = os.environ["DB_USERNAME"]

data_path = os.environ["URL"] + os.environ["DATA"]

engine = create_engine(f'postgresql+psycopg://{db_us}:{db_pw}@{db_host}:{db_port}/{db_name}')
# Read a sample of the data
dtype = {
    "VendorID": "Int64",
    "passenger_count": "Int64",
    "trip_distance": "float64",
    "RatecodeID": "Int64",
    "store_and_fwd_flag": "string",
    "PULocationID": "Int64",
    "DOLocationID": "Int64",
    "payment_type": "Int64",
    "fare_amount": "float64",
    "extra": "float64",
    "mta_tax": "float64",
    "tip_amount": "float64",
    "tolls_amount": "float64",
    "improvement_surcharge": "float64",
    "total_amount": "float64",
    "congestion_surcharge": "float64"
}

parse_dates = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime"
]

df_iter = pd.read_csv(
    data_path,
    dtype=dtype,
    parse_dates=parse_dates,
    iterator=True,
    chunksize=100000,
)
total_rows = 0
for i, df_chunk in enumerate(tqdm(df_iter, desc="Ingestion progression")):
    
    mode = "replace" if i == 0 else "append"
    df_chunk.to_sql(name="yellow_taxi_data", con=engine, if_exists=mode, index=False)
    total_rows += len(df_chunk)
    print("Inserted:", total_rows)

count = pd.read_sql("SELECT COUNT(*) FROM yellow_taxi_data;", con=engine).iloc[0, 0]
print(total_rows == count)

Ingestion progression: 0it [00:00, ?it/s]

Inserted: 100000
Inserted: 200000
Inserted: 300000
Inserted: 400000
Inserted: 500000
Inserted: 600000
Inserted: 700000
Inserted: 800000
Inserted: 900000
Inserted: 1000000
Inserted: 1100000
Inserted: 1200000
Inserted: 1300000
Inserted: 1369765
True


df_csv = pd.read_csv(
    data_path,
    dtype=dtype,
    parse_dates=parse_dates)

print(len(df_csv))
print(df_csv["passenger_count"].sum())
print(df_csv["tpep_pickup_datetime"].max())
print(df_csv["tpep_pickup_datetime"].min())